# Install these dependencies 

In [ ]:
pip install pandas
pip install spacy
pip install tqdm
pip install swifter
pip install matplotlib
pip install scikit-learn
pip install numpy
pip install sentence-transformers
pip install umap-learn
pip install yake
pip install langchain-community
pip install openpyxl


# Import Libraries 

In [ ]:
import pandas as pd
import spacy
from tqdm import tqdm
import swifter
import matplotlib.pyplot as plt
from langchain_community.llms import Ollama
from sklearn.cluster import KMeans
import numpy as np
import re
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import umap
from sklearn.cluster import KMeans
from yake import KeywordExtractor
from langchain_community.llms import Ollama
import umap



# Importing, Dataset

In [ ]:
# Download the data from google drive link 

In [ ]:

Data_Download_Link_Google_Drive: "https://docs.google.com/spreadsheets/d/1YGWMIlIOeeTMTIrvvzPR3Xjv0g7zBWBu/edit?usp=drive_link&ouid=116245445694931910291&rtpof=true&sd=true"


In [ ]:
# Set the path of data folder 

In [ ]:
# Please select your data path 
path_to_your_data_folder = "New_ProjectMapeData.xlsx"
df = pd.read_excel(path_to_your_data_folder)


# Topic Modeling , using LLM 

In [ ]:
def load_dataset_with_progress(file_path="modified_letters.csv"):
    df = pd.read_csv(file_path)
    df['LLM Suggested Topic'] = None  
    print(f"Starting fresh with the dataset. Loaded '{file_path}'.")
    return df

def load_llm():
    llm = Ollama(model="llama3:latest")  
    return llm

def extract_topic_from_letter(llm, text):
    prompt = (
        f"Given the following historical letter, generate a concise topic for cataloging purposes. "
        f"Ignore any references to dates, people’s names, or locations. "
        f"Focus only on the main topic of the letter. Keep it short and focused on the subject discussed:\n\n"
        f"{text}\n\n"
        f"Main Topic:"
    )
    topic = llm.predict(prompt).strip()
    
    # topic = re.sub(r'\b\d{4}\b', '', topic)  
    return topic

def process_and_save(df, file_path="letters_with_clustered_topics.csv"):
    llm = load_llm()

    for i, row in df.iterrows():
        if pd.isna(row['LLM Suggested Topic']):  
            letter_text = row['doc_text']
            topic = extract_topic_from_letter(llm, letter_text)
            df.at[i, 'LLM Suggested Topic'] = topic  

            print(f"Letter {i+1}:\n{letter_text}")
            print(f"Concise Topic: {topic}\n")

            df.to_csv(file_path, index=False)
            print(f"Progress saved to '{file_path}'.")

    return df

def main():
    df = load_dataset_with_progress()

    df_with_topics = process_and_save(df)

    df_with_topics.to_csv("letters_with_topics_final.csv", index=False)

main()


In [ ]:
# Inspect the Final File After LLM 
# Remove Errors (such as Null Response, Errors)

# Data File After LLM Cycle

In [ ]:
Google_Drive_Link = "https://docs.google.com/spreadsheets/d/1keeGE1piobtxPEXGjY03Wq0o-h5NPo5E/edit?usp=drive_link&ouid=116245445694931910291&rtpof=true&sd=true"
df_after_llm = pd.read_excel("projectmapedata_afterLLM.xlsx")
df_after_llm

In [ ]:
df_after_llm.shape

# Embedding Generation-(Doc-Text and LLM Topics) & Topic Validation against LLM suggested topic

In [ ]:
model = SentenceTransformer('sentence-transformers/paraphrase-xlm-r-multilingual-v1', device='cuda')

doc_text_embeddings = model.encode(
    df_after_llm['doc_text'].tolist(), 
    show_progress_bar=True, 
    batch_size=32  
)

topic_embeddings = model.encode(
    df_after_llm['LLM Suggested Topic'].tolist(), 
    show_progress_bar=True, 
    batch_size=32  
)

np.save("doc_text_embeddings.npy", doc_text_embeddings)
np.save("topic_embeddings.npy", topic_embeddings)

print("Embeddings saved successfully!")

doc_text_embeddings = np.load("doc_text_embeddings.npy")
topic_embeddings = np.load("topic_embeddings.npy")
cosine_similarities = [
    cosine_similarity([doc_text_embeddings[i]], [topic_embeddings[i]])[0][0]
    for i in range(len(doc_text_embeddings))
]

df_after_llm['Cosine_Similarity'] = cosine_similarities

mean = df_after_llm['Cosine_Similarity'].mean()
std = df_after_llm['Cosine_Similarity'].std()

plt.figure(figsize=(10, 6))
plt.hist(df_after_llm['Cosine_Similarity'], bins=30, edgecolor='black', alpha=0.7)

plt.axvline(mean, color='blue', linestyle='--', linewidth=1.5, label='Mean')
plt.axvline(mean + 1 * std, color='green', linestyle='--', linewidth=1.5, label='+1 Sigma')
plt.axvline(mean - 1 * std, color='green', linestyle='--', linewidth=1.5, label='-1 Sigma')
plt.axvline(mean + 2 * std, color='orange', linestyle='--', linewidth=1.5, label='+2 Sigma')
plt.axvline(mean - 2 * std, color='orange', linestyle='--', linewidth=1.5, label='-2 Sigma')
plt.axvline(mean + 3 * std, color='red', linestyle='--', linewidth=1.5, label='+3 Sigma')
plt.axvline(mean - 3 * std, color='red', linestyle='--', linewidth=1.5, label='-3 Sigma')

plt.title('Histogram of Cosine Similarity with Standard Deviation Lines', fontsize=16)
plt.xlabel('Cosine Similarity', fontsize=14)
plt.ylabel('Frequency', fontsize=14)

plt.legend(fontsize=12, loc='upper left')
plt.savefig('histogram_cosine_similarity.png', dpi=500, bbox_inches='tight')

plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()


 # Clustering and Labeling Process 

In [ ]:
combined_embeddings = np.hstack((doc_text_embeddings, topic_embeddings))
filtered_df = df_after_llm[df_after_llm['Cosine_Similarity'] >= 0.3].reset_index(drop=True)
filtered_indices = filtered_df.index
filtered_embeddings = combined_embeddings[filtered_indices]
reducer_50d = umap.UMAP(n_components=50, 
                        random_state=42,
                        verbose=True)
filtered_umap_embeddings_50d = reducer_50d.fit_transform(filtered_embeddings)

filtered_umap_df = pd.DataFrame(
    filtered_umap_embeddings_50d, 
    columns=[f"UMAP_{i+1}" for i in range(50)]
)

filtered_final_df = pd.concat([filtered_df, 
                               filtered_umap_df], axis=1)
reducer_2d = umap.UMAP(n_components=2, random_state=42, verbose=True)
embeddings_2d = reducer_2d.fit_transform(filtered_embeddings)

umap_embeddings = filtered_final_df.iloc[:, 7:]

k_values = range(4, 15)
inertia_scores = []

for k in k_values:
    kmeans = KMeans(n_clusters=k, random_state=42)
    kmeans.fit(umap_embeddings)  
    inertia_scores.append(kmeans.inertia_)  

plt.figure(figsize=(10, 6))
plt.plot(k_values, inertia_scores, marker='o')
plt.axvline(x=8, color='red', linestyle='--', label='k=8 Elbow')
plt.title('Elbow Method for Optimal k')
plt.xlabel('Number of Clusters (k)')
plt.ylabel('Inertia (Sum of Squared Distances)')
plt.xticks(k_values)
plt.grid(True)
plt.show()
def perform_clustering(umap_embeddings, 
                       num_clusters=8):
    kmeans = KMeans(n_clusters=num_clusters,
                    random_state=42)
    cluster_labels = kmeans.fit_predict(umap_embeddings)
    return cluster_labels

cluster_labels = perform_clustering(filtered_final_df.iloc[:, 7:],
                                    num_clusters=8)
filtered_final_df["Cluster_Labels"] = cluster_labels
def preprocess_topics(df, column_name, new_column_name):
    nlp = spacy.load("en_core_web_sm")
    stopwords = nlp.Defaults.stop_words

    def preprocess(text):
        doc = nlp(text)
        tokens = [token.lemma_.lower() for token in doc if token.is_alpha and token.text.lower() not in stopwords]
        return " ".join(tokens)

    tqdm.pandas(desc="Preprocessing topics")
    df[new_column_name] = df[column_name].progress_apply(preprocess)
    return df

filtered_final_df = preprocess_topics(filtered_final_df, 'LLM Suggested Topic', 'Processed LLM Suggested Topic')
def extract_keywords_per_cluster(df, cluster_labels, num_clusters=10):

    df['Cluster_Label'] = cluster_labels
    
    keyword_extractor_no_stopwords = KeywordExtractor(
        lan="english",
        n=2, 
        dedupLim=0.9, 
        top=50,
        windowsSize=1
    )
    
    keyword_extractor_with_stopwords = KeywordExtractor(
        lan="english",
        n=2, 
        dedupLim=0.3,  
        top=50,
        windowsSize=2,
        stopwords=["request", "confirmation", "military", "governance", 
                   "appointment", "colonial", "rank", "official"]
    )
    
    keywords_before_stopwords = {}
    keywords_after_stopwords = {}

    for cluster in tqdm(range(num_clusters), desc="Extracting keywords for clusters"):
        cluster_texts = df[df['Cluster_Label'] == cluster]['Processed LLM Suggested Topic'].tolist()
        combined_text = " ".join(cluster_texts)
        
        keywords_no_stopwords = [kw[0] for kw in keyword_extractor_no_stopwords.extract_keywords(combined_text)]
        keywords_before_stopwords[cluster] = keywords_no_stopwords
        
        keywords_with_stopwords = [kw[0] for kw in keyword_extractor_with_stopwords.extract_keywords(combined_text)]
        keywords_after_stopwords[cluster] = keywords_with_stopwords
        
        print(f"\nCluster {cluster}:")
        print(f"Keywords Before Stopword Removal: {keywords_no_stopwords}")
        print(f"Keywords After Stopword Removal: {keywords_with_stopwords}")
    
    return keywords_before_stopwords, keywords_after_stopwords
keywords_before, keywords_after = extract_keywords_per_cluster(
    filtered_final_df,
    cluster_labels,
    num_clusters=8
)




In [ ]:
# Extracted Keywords as per each cluster helped in labelling each Cluster 

In [ ]:
final_Data = pd.read_excel("FinalDataToAfterAllStages_PNC.xlsx")